# Workshop 3B: Digital Certificates & PKI Lab

**Topic**: Concurrency, Operating Systems, & Security — Digital Certificates & Public Key Infrastructure (PKI)  
**Target Duration**: ~20 minutes  
**Objective**: Programmatically retrieve, inspect, and verify SSL/TLS certificates using Python's built-in `socket` and `ssl` libraries, understanding host verification and certificate chain validation.

---  
## Exercise 1: Programmatic SSL/TLS Certificate Retrieval

To establish a secure HTTPS connection, the client and server perform a TLS handshake. In this step, the client downloads the server's public digital certificate and validates it against its Root Store.

We will connect to `london.port.ac.uk` on port `443` to retrieve and inspect its certificate.

In [ ]:
import socket
import ssl

hostname = "london.port.ac.uk"
port = 443

# 1. Create a default SSL context (loads default Root certificates from the operating system)
context = ssl.create_default_context()

# 2. Establish connection and wrap the socket in SSL
with socket.create_connection((hostname, port)) as sock:
    with context.wrap_socket(sock, server_hostname=hostname) as ssock:
        # TODO: Retrieve the parsed certificate dictionary from the SSL socket
        # Hint: Call the getpeercert() method on the wrapped socket 'ssock'
        cert = ...

# 3. Parse and Print metadata
if cert:
    # Convert subject and issuer tuple format into a readable dictionary
    subject_dict = dict(x[0] for x in cert.get('subject', []))
    issuer_dict = dict(x[0] for x in cert.get('issuer', []))
    
    print(f"--- Certificate Details for {hostname} ---")
    print(f"Common Name (Domain): {subject_dict.get('commonName')}")
    print(f"Issuer CA:            {issuer_dict.get('commonName')}")
    
    # TODO: Print the 'notBefore' (valid from) and 'notAfter' (valid until) fields from the cert dictionary
    # Hint: Use cert.get('notBefore') and cert.get('notAfter')
    print(f"Valid From:           ...")
    print(f"Valid Until:          ...")
else:
    print("Could not retrieve certificate.")

---  
## Exercise 2: Extracting Subject Alternative Names (SANs) and Serial Number

A single SSL certificate can protect multiple subdomains or domains using the **Subject Alternative Name (SAN)** field. In this task, you will programmatically parse and list all alternative domains covered by the certificate.

In [ ]:
# TODO: Extract the Serial Number and Subject Alternative Names (SANs) from the 'cert' dictionary
# Hint 1: The serial number is a hexadecimal string stored under the key 'serialNumber'
# Hint 2: The SANs are stored under 'subjectAltName' as a sequence of tuples: (('DNS', 'domain1'), ('DNS', 'domain2'))
#         Retrieve it using cert.get('subjectAltName', []) and iterate over the elements

serial_number = ...
sans = ...

print(f"Certificate Serial Number: {serial_number}")
print("\nSubject Alternative Names (SANs):")
for san_type, san_value in sans:
    print(f"  - {san_type}: {san_value}")

---  
## Exercise 3: Observing SSL/TLS Verification Failures (Self-Signed Certificates)

If a server provides a certificate that is not signed by a CA in our trusted Root Store, or if the domain name doesn't match the certificate, the browser/client will reject the connection.

Let's test this by attempting to connect to `self-signed.badssl.com`.

In [ ]:
bad_hostname = "self-signed.badssl.com"

# TODO: Write a try-except block to connect to bad_hostname on port 443
# Catch the ssl.SSLCertVerificationError and print the verification failure explanation
# Hint: Use socket.create_connection and context.wrap_socket as before

context = ssl.create_default_context()

try:
    ...
except ssl.SSLCertVerificationError as e:
    print(f"Verification Failed (Expected!): {e}")
    print("Reason: The certificate presented is self-signed and not issued by a trusted root CA.")
except Exception as e:
    print(f"Other Connection Error: {e}")

---  
## Exercise 4: Automated Expiration Check Alerting Script

SSL certificates must be renewed frequently. Modern system administrators write monitoring scripts that parse certificate expiration dates and alert if a certificate is close to expiring.

Write a function `check_days_until_expiration(cert_dict)` that computes the remaining days before a certificate becomes invalid.

In [ ]:
from datetime import datetime

def check_days_until_expiration(cert_dict):
    not_after_str = cert_dict.get('notAfter')
    if not not_after_str:
        return None
    
    # TODO: Convert the 'notAfter' string date to a python datetime object
    # Hint 1: The date format returned by Python ssl is like: 'Oct  7 12:00:00 2026 GMT'
    #         Use datetime.strptime(not_after_str, "%b %d %H:%M:%S %Y %Z")
    # Hint 2: Calculate the delta between the expiry datetime and the current datetime: expiry - datetime.utcnow()
    #         Return the delta.days field as an integer
    expiry_date = ...
    days_remaining = ...
    
    return days_remaining

# Test using the certificate retrieved in Exercise 1
days_left = check_days_until_expiration(cert)
print(f"Days remaining until certificate expiration: {days_left} days")
if days_left < 30:
    print("WARNING: Certificate expires in less than 30 days! Action required.")
else:
    print("Certificate status: Active and secure.")

### Discussion Questions
1. **Why is host name verification (matching the URL host to the certificate Common Name/SAN) a critical part of the TLS handshake?**
2. **Why do certificates have short expiration dates (typically 397 days maximum today)?**